# Calculus Warm-up for Causal Inference
**2-day sprint · derivatives → integrals → probability**

Run every cell top to bottom. By the end you are ready for Blitzstein Week 1.

---

In [ ]:
# Setup — run this first
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.facecolor'] = '#f8f8f8'

print('Libraries loaded. Ready to go.')

---
## DAY 1 — Derivatives

A derivative measures how fast a function changes at a point. Geometrically: **the slope of the curve**.

$$f'(x) = \\lim_{h \\to 0} \\frac{f(x+h) - f(x)}{h}$$

> **Why this matters for causal inference:** estimating treatment effects means maximising a likelihood. Maximising = finding where the derivative = 0.

**Power rule:** if $f(x) = x^n$ then $f'(x) = n \\cdot x^{n-1}$

In [ ]:
# Cell 1: Power rule verification
# f(x) = 3x^2 + 2x  ->  f'(x) = 6x + 2

def f(x):
    return 3*x**2 + 2*x

def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x)) / h

x = 4
df_analytical = 6*x + 2
df_numerical  = numerical_derivative(f, x)

print(f'f(x) = 3x^2 + 2x   at x = {x}')
print(f"f'(x) analytical = {df_analytical}")
print(f"f'(x) numerical  = {df_numerical:.4f}")
print(f'Match? {abs(df_numerical - df_analytical) < 0.01}')

### The chain rule

$$\\frac{d}{dx}[f(g(x))] = f'(g(x)) \\cdot g'(x)$$

> **Why this matters:** backpropagation in neural networks is the chain rule applied recursively. Also used when differentiating log-likelihoods for causal estimators.

Example: $\\frac{d}{dx}[e^{x^2}] = e^{x^2} \\cdot 2x$

In [ ]:
# Cell 2: Chain rule
# d/dx [log(x^2 + 1)]
# Outer: log(u) -> 1/u
# Inner: x^2 + 1 -> 2x
# Result: 2x / (x^2 + 1)

import math

def g(x):
    return math.log(x**2 + 1)

def g_prime_analytical(x):
    return 2*x / (x**2 + 1)

def g_prime_numerical(x, h=1e-7):
    return (g(x + h) - g(x)) / h

x = 3.0
print('d/dx[log(x^2 + 1)]  at x = 3')
print(f'Analytical (chain rule): {g_prime_analytical(x):.6f}')
print(f'Numerical approx:        {g_prime_numerical(x):.6f}')
print()
print('Log derivatives appear everywhere in')
print('log-likelihood estimation for causal models.')

In [ ]:
# Cell 3: Finding the maximum (where f'(x) = 0)
# This is what MLE does to find a causal effect estimate.
#
# f(x) = -x^2 + 4x + 1
# f'(x) = -2x + 4  ->  set to 0  ->  x = 2

import numpy as np
import matplotlib.pyplot as plt

def f(x):
    return -x**2 + 4*x + 1

def f_prime(x, h=1e-7):
    return (f(x + h) - f(x)) / h

xs   = np.linspace(0, 5, 1000)
vals = f(xs)
max_x = xs[np.argmax(vals)]

print('f(x) = -x^2 + 4x + 1')
print(f'Analytical max at x = 2       f(2) = {f(2)}')
print(f'Numerical  max at x = {max_x:.3f}    f   = {f(max_x):.4f}')
print(f"f'(2) = {f_prime(2):.6f}   (should be ~0)")
print()
print("Setting f'(x)=0 is how MLE finds a causal effect estimate.")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
xs_plot = np.linspace(-1, 5, 300)
ax1.plot(xs_plot, f(xs_plot), color='#7c6af7', linewidth=2.5, label='f(x)')
ax1.axvline(x=2, color='#f7b267', linestyle='--', alpha=0.8, label='max at x=2')
ax1.scatter([2], [f(2)], color='#f7b267', zorder=5, s=80)
ax1.set_title('f(x) = -x^2 + 4x + 1'); ax1.legend()
ax2.plot(xs_plot, f_prime(xs_plot), color='#4ecdc4', linewidth=2.5, label="f'(x)")
ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax2.axvline(x=2, color='#f7b267', linestyle='--', alpha=0.8, label="f'(x)=0 at x=2")
ax2.set_title("f'(x) = -2x + 4"); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Cell 4: Plot sin(x) and its derivative cos(x)
# Where sin peaks, cos = 0  ->  derivative = 0 at maximum

import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-2*np.pi, 2*np.pi, 500)
plt.figure(figsize=(11, 4))
plt.plot(x, np.sin(x), color='#7c6af7', linewidth=2.5, label='sin(x)')
plt.plot(x, np.cos(x), color='#4ecdc4', linewidth=2, linestyle='--', label="cos(x) = sin'(x)")
plt.axhline(0, color='gray', linewidth=0.8, alpha=0.5)
for px in [-3*np.pi/2, np.pi/2]:
    plt.axvline(px, color='#f7b267', alpha=0.4, linewidth=1)
plt.title('sin(x) and its derivative cos(x) -- where sin peaks, cos = 0')
plt.xlabel('x'); plt.legend()
plt.tight_layout(); plt.show()
print('Orange lines mark peaks of sin(x). At those points cos(x) = 0.')

---
## DAY 2 — Integrals

A definite integral computes the **area under a curve** between two points.

$$\\int_a^b f(x)\\, dx = F(b) - F(a) \\quad \\text{where } F'(x) = f(x)$$

> **Why this matters for probability:**
> $$P(a \\leq X \\leq b) = \\int_a^b f(x)\\, dx$$
> You will write this every week in Blitzstein.

Basic rule: $\\int x^n\\, dx = \\frac{x^{n+1}}{n+1} + C$

In [ ]:
# Cell 5: Numerical integration (Riemann sums)
# This is what scipy.integrate does under the hood.

import numpy as np

def riemann_integral(f, a, b, n=100_000):
    h = (b - a) / n
    xs = np.linspace(a, b - h, n)
    return np.sum(f(xs) * h)

r1 = riemann_integral(lambda x: 2*x, 0, 1)
r2 = riemann_integral(lambda x: x**2, 0, 2)

print('Integral 1: integral from 0 to 1 of 2x dx')
print(f'  Analytical: 1.000000')
print(f'  Numerical:  {r1:.6f}')
print()
print('Integral 2: integral from 0 to 2 of x^2 dx')
print(f'  Analytical: {8/3:.6f}')
print(f'  Numerical:  {r2:.6f}')

In [ ]:
# Cell 6: Integrals = Probabilities  (the key connection)
#
# Uniform[0,1]: f(x) = 1 for x in [0,1]
# P(0.2 <= X <= 0.7) = integral from 0.2 to 0.7 of 1 dx = 0.5

import numpy as np
import matplotlib.pyplot as plt

def riemann_integral(f, a, b, n=100_000):
    h = (b - a) / n
    xs = np.linspace(a, b - h, n)
    return np.sum(f(xs) * h)

def uniform_pdf(x):
    return np.where((x >= 0) & (x <= 1), 1.0, 0.0)

total = riemann_integral(uniform_pdf, 0, 1)
prob  = riemann_integral(uniform_pdf, 0.2, 0.7)

print(f'Total probability: {total:.6f}  (should be 1.0)')
print(f'P(0.2 <= X <= 0.7): {prob:.6f}  (should be 0.5)')
print()
print('Every continuous probability is an integral.')

x = np.linspace(-0.5, 1.5, 400)
plt.figure(figsize=(8, 3.5))
plt.plot(x, uniform_pdf(x), color='#7c6af7', linewidth=2.5)
x_shade = np.linspace(0.2, 0.7, 200)
plt.fill_between(x_shade, uniform_pdf(x_shade), alpha=0.35, color='#7c6af7',
                 label='P(0.2 <= X <= 0.7) = 0.5')
plt.ylim(-0.1, 1.5)
plt.title('Uniform[0,1] PDF -- shaded area = probability')
plt.xlabel('x'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Cell 7: Normal distribution -- 68-95-99.7 rule
# This is where p-values in A/B tests come from.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def riemann_integral(f, a, b, n=100_000):
    h = (b - a) / n
    xs = np.linspace(a, b - h, n)
    return np.sum(f(xs) * h)

def normal_pdf(x, mu=0, sigma=1):
    return (1 / (sigma * np.sqrt(2*np.pi))) * np.exp(-0.5 * ((x - mu)/sigma)**2)

p1 = riemann_integral(lambda x: normal_pdf(x), -1, 1)
p2 = riemann_integral(lambda x: normal_pdf(x), -2, 2)
p3 = riemann_integral(lambda x: normal_pdf(x), -3, 3)

print('Normal(mu=0, sigma=1) -- 68-95-99.7 rule:')
print(f'P(-1s <= X <= 1s) = {p1:.4f}   (expected ~0.6827)')
print(f'P(-2s <= X <= 2s) = {p2:.4f}   (expected ~0.9545)')
print(f'P(-3s <= X <= 3s) = {p3:.4f}   (expected ~0.9973)')
print()
print('P(|Z| > 1.96) ~ 0.05  ->  the 5% significance level in A/B tests.')

x = np.linspace(-4, 4, 500)
y = normal_pdf(x)
plt.figure(figsize=(10, 4))
plt.plot(x, y, color='#7c6af7', linewidth=2.5)
for a, b, col, alpha in [(-1,1,'#7c6af7',0.35),(-2,-1,'#4ecdc4',0.25),(1,2,'#4ecdc4',0.25),
                          (-3,-2,'#f7b267',0.2),(2,3,'#f7b267',0.2)]:
    xf = np.linspace(a, b, 200)
    plt.fill_between(xf, normal_pdf(xf), alpha=alpha, color=col)
plt.axvline(-1.96, color='#f7706e', linestyle='--', alpha=0.7, linewidth=1.2)
plt.axvline( 1.96, color='#f7706e', linestyle='--', alpha=0.7, linewidth=1.2)
plt.text(2.0, 0.3, '1.96\n(p=0.05)', color='#f7706e', fontsize=9)
patches = [mpatches.Patch(color='#7c6af7', alpha=0.6, label='+-1s  68.3%'),
           mpatches.Patch(color='#4ecdc4', alpha=0.5, label='+-2s  95.4%'),
           mpatches.Patch(color='#f7b267', alpha=0.5, label='+-3s  99.7%')]
plt.legend(handles=patches, loc='upper left')
plt.title('Normal(0,1) -- the distribution behind p-values')
plt.xlabel('Standard deviations'); plt.tight_layout(); plt.show()

In [ ]:
# Cell 8: E[X] and Var[X] via integration
# E[X]   = integral of x * f(x) dx
# Var[X] = E[X^2] - (E[X])^2
# For Uniform[0,1]: E[X] = 0.5,  Var[X] = 1/12 ~ 0.0833

import numpy as np

def riemann_integral(f, a, b, n=100_000):
    h = (b - a) / n
    xs = np.linspace(a, b - h, n)
    return np.sum(f(xs) * h)

def uniform_pdf(x):
    return np.where((x >= 0) & (x <= 1), 1.0, 0.0)

EX   = riemann_integral(lambda x: x * uniform_pdf(x),    0, 1)
EX2  = riemann_integral(lambda x: x**2 * uniform_pdf(x), 0, 1)
VarX = EX2 - EX**2

print(f'E[X]   = {EX:.6f}   (should be 0.5)')
print(f'Var[X] = {VarX:.6f}  (should be 1/12 = {1/12:.6f})')
print()
print('=' * 45)
print('  WARM-UP COMPLETE. Ready for Week 1.  ')
print('=' * 45)
print()
print('You can now:')
print('  - Compute derivatives  (slopes, maxima)')
print('  - Compute integrals    (areas, probabilities)')
print('  - Link integrals to E[X] and P(a <= X <= b)')
print()
print('Open Blitzstein ch.1 on Monday.')

---
## Quick reference

| Concept | Formula | Used for |
|---|---|---|
| Derivative | $f'(x) = \\lim_{h\\to0}\\frac{f(x+h)-f(x)}{h}$ | Finding maxima of likelihoods |
| Power rule | $\\frac{d}{dx}x^n = nx^{n-1}$ | Differentiating polynomials |
| Chain rule | $\\frac{d}{dx}f(g(x)) = f'(g(x))\\cdot g'(x)$ | Log-likelihoods, backprop |
| Integral | $\\int_a^b f(x)\\,dx = F(b)-F(a)$ | Computing probabilities |
| Expectation | $E[X] = \\int x\\,f(x)\\,dx$ | Average treatment effect |
| Variance | $Var[X] = E[X^2] - (E[X])^2$ | Uncertainty in estimates |

**Next:** Blitzstein & Hwang — *Introduction to Probability*, chapters 1-7.